# Lab 1 Generate test data: Custoemr list. Show Spark Merge operation

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("Lab-1_Preparation") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.warehouse.dir", "lab_1/lakehouse") 

spark = configure_spark_with_delta_pip(builder).getOrCreate()
print(f"Spark version: {spark.version} with Delta support")

Spark version: 3.5.0 with Delta support


## 1. Prepare Test Data

In [4]:
from faker import Faker
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
import pandas as pd

fake = Faker()

print("Generate test data........")
# 1. generate fake data
def generate_fake_data(num_records):
    data = []
    for _ in range(num_records):
        data.append({
            "client_id": fake.uuid4(),
            "name": fake.name(),
            "email": fake.email(),
            "city": fake.city(),
            "balance": fake.random_int(min=0, max=100000)
        })
    return data

# 2. Create Spark DataFrame
raw_data = generate_fake_data(1000) # generate 1000 records
df = spark.createDataFrame(pd.DataFrame(raw_data))

# 3. Write to Lab-1 Warehouse
df.write.format("delta").mode("overwrite").saveAsTable("bronze_clients")

print("Generate test data - FINISHED")

Generate test data........
Generate test data - FINISHED


In [5]:
spark.sql("SELECT * FROM bronze_clients").show()

+--------------------+-----------------+--------------------+-------------------+-------+
|           client_id|             name|               email|               city|balance|
+--------------------+-----------------+--------------------+-------------------+-------+
|968dc41e-f2cb-4e7...|  Heather Anthony|estesrenee@exampl...|         Lake Randy|  78866|
|b9ed8080-56f6-404...|   Robert Navarro| brian36@example.com|       Salazarburgh|   4614|
|d75a6235-9313-49d...|    Sandra Krause|  fstout@example.org|        West Joseph|  27017|
|aa2a9cae-cb33-498...|     Makayla Hill|stevenroberts@exa...|        Roberthaven|  88784|
|5d6ee016-2277-4a9...|    Henry Mueller|sandra85@example.net|  East Charlesburgh|  65273|
|6f4ac06b-8dbd-4a5...|Breanna Hernandez|michellelopez@exa...|Port Christinamouth|  86448|
|97b6704b-39a4-459...|      Lisa Martin|kennethandersen@e...|     Port Ericmouth|  78492|
|efc34792-81a8-43d...|  Matthew Fischer|webbstephen@examp...|   West Brookeville|  44039|
|675779a4-

##  2. Generate "delta" (Updates & Inserts)

We will take some IDs from the existing table and create new balance and city values ​​for them, as well as add new people.

In [7]:
from faker import Faker
import pandas as pd

fake = Faker()

print("Processing Start........")
# 1. Get IDs for update existing records
existing_ids = [row.client_id for row in spark.sql("SELECT client_id FROM bronze_clients LIMIT 5").collect()]

# 2. Generate data for Merge (changed exists recirs + new records)
merge_data = []

# Update for exists records
for cid in existing_ids:
    merge_data.append({
        "client_id": cid,
        "name": fake.name(), # припустимо, зміна імені/прізвища
        "email": fake.email(),
        "city": "UPDATED_CITY",
        "balance": 99999
    })

# New customers
for _ in range(3):
    merge_data.append({
        "client_id": fake.uuid4(),
        "name": fake.name(),
        "email": fake.email(),
        "city": fake.city(),
        "balance": fake.random_int(min=1000, max=5000)
    })

updates_df = spark.createDataFrame(pd.DataFrame(merge_data))
updates_df.createOrReplaceTempView("updates_view")
print("Processig - FINISHED")

Processing Start........
Processig - FINISHED


In [8]:
updates_df.show()

+--------------------+---------------+--------------------+-----------------+-------+
|           client_id|           name|               email|             city|balance|
+--------------------+---------------+--------------------+-----------------+-------+
|968dc41e-f2cb-4e7...|    Sean Murray|vasquezmike@examp...|     UPDATED_CITY|  99999|
|b9ed8080-56f6-404...|Michelle Mccall|noblemichael@exam...|     UPDATED_CITY|  99999|
|d75a6235-9313-49d...|Kevin Blackburn|walkerdarrell@exa...|     UPDATED_CITY|  99999|
|aa2a9cae-cb33-498...|Jennifer Tucker| kgibson@example.com|     UPDATED_CITY|  99999|
|5d6ee016-2277-4a9...|   Robert Moore|wilsonleslie@exam...|     UPDATED_CITY|  99999|
|c1f5bf7a-3da8-4a1...|     Keith Hall|  bperry@example.com|South Christopher|   3307|
|9e2dab25-d736-4d3...|   Molly Foster|patriciaellison@e...|    Martinezmouth|   1583|
|cdb72e6c-4afc-4c1...|   Colleen Beck|paulthompson@exam...|   West Coreyfort|   3326|
+--------------------+---------------+----------------

## 3. Merge changes into main table

In [10]:
spark.sql("""
MERGE INTO bronze_clients AS target
USING updates_view AS source
ON target.client_id = source.client_id
WHEN MATCHED THEN
  UPDATE SET 
    target.name = source.name,
    target.balance = source.balance,
    target.city = source.city
WHEN NOT MATCHED THEN
  INSERT *
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

## 4. View History and TimeTravel

In [12]:
# Check history
from delta.tables import DeltaTable
deltaInstance = DeltaTable.forName(spark, "bronze_clients")
deltaInstance.history().select("version", "timestamp", "operation", "operationMetrics").show(truncate=False)

# Time Travel: look, how it was defore update (version 0)
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("bronze_clients")
print("Customer balanse before MERGE:")
df_v0.filter(df_v0.client_id == existing_ids[0]).show()

print("Current balance of the same customer (версія 1):")
spark.sql(f"SELECT * FROM bronze_clients WHERE client_id = '{existing_ids[0]}'").show()

+-------+-----------------------+---------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp              |operation                        |operationMetrics                                                                                                                                                                                                                      